# Chapitre 9 - Exercices 9.1 à 9.3 : lisseurs linéaires et backfitting

Ces trois exercices étudient le backfitting (algorithme 9.1 du livre) sous l'angle de
l'**algèbre linéaire** : quand les lisseurs sont linéaires, $\hat{\mathbf f}_j = S_j\,
\mathbf r$, l'algorithme devient une méthode itérative classique de résolution d'un
système linéaire.

- **Exercice 9.1** : une spline de lissage préserve la partie linéaire de l'ajustement ;
  conséquence pour le recentrage dans le backfitting.
- **Exercice 9.2** : le backfitting est un algorithme de Gauss–Seidel par blocs ;
  convergence pour deux lisseurs symétriques.
- **Exercice 9.3** : avec des projections orthogonales, les équations du backfitting
  sont les équations normales des moindres carrés.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl.ch09_arbres import (SplineLisseur, MatriceLisseur, local_linear_matrix,
                             backfitting)

set_style()
rng = np.random.RandomState(0)

## Exercice 9.1 : La spline de lissage préserve la partie linéaire

**Énoncé (reformulé).** Décomposons la réponse en $y_i = \hat y_i + r_i$, où $\hat y$
est l'ajustement de la régression linéaire simple de $y$ sur $x$ et $r$ le résidu.
Soit $S$ la matrice de lissage d'une spline de lissage. Montrer que
$$S y = \hat y + S r,$$
c'est-à-dire que la spline reproduit exactement la partie linéaire. Montrer qu'il en va
de même pour la régression linéaire locale (section 6.1.1). En déduire que l'étape de
recentrage (seconde ligne de l'étape 2 de l'algorithme 9.1) est inutile.

### Rappels

- **Spline de lissage** (section 5.4) : $\hat f$ minimise $\sum_i(y_i-f(x_i))^2 +
  \lambda\int f''^2$. Dans une base $B_1,\dots,B_K$ de splines contenant les fonctions
  affines, $f=\sum_k c_kB_k$, on a $\hat c = (B^TB+\lambda\Omega)^{-1}B^Ty$ et
  $S=B(B^TB+\lambda\Omega)^{-1}B^T$, avec $\Omega_{kl}=\int B_k''B_l''$.
- **Régression linéaire locale** (éq. (6.7)-(6.9)) : en chaque point $x_0$, on résout
  $\min_{\alpha,\beta}\sum_i K_\lambda(x_0,x_i)\,(y_i-\alpha-\beta x_i)^2$ et l'on pose
  $\hat f(x_0)=\hat\alpha(x_0)+\hat\beta(x_0)x_0$. Matriciellement, avec $\mathbf B$ la
  matrice $N\times 2$ de lignes $(1,x_i)$ et $W(x_0)=\mathrm{diag}(K_\lambda(x_0,x_i))$ :
  $\hat f(x_0) = (1,x_0)\,(\mathbf B^TW(x_0)\mathbf B)^{-1}\mathbf B^TW(x_0)\,y$.

### Solution : le cas de la spline de lissage

Le vecteur $\hat y$ est de la forme $\hat y_i = a + b x_i$ : ce sont les valeurs aux
$x_i$ de la fonction affine $\ell(x)=a+bx$.

**Étape 1 : $S\hat y = \hat y$.** Appliquons la spline de lissage aux données
$(x_i,\hat y_i)$. Pour la fonction $\ell$ :
- le terme d'ajustement $\sum_i(\hat y_i-\ell(x_i))^2$ vaut $0$ ;
- le terme de pénalité $\lambda\int\ell''^2$ vaut $0$ car $\ell''=0$.

Le critère étant toujours $\ge 0$, $\ell$ en est un minimiseur ; le critère étant
strictement convexe (pour $\lambda>0$ et au moins deux valeurs distinctes de $x$), c'est
**le** minimiseur. Donc la spline ajustée à $\hat y$ est $\ell$ elle-même : $S\hat y =
\hat y$.

**Étape 2 : linéarité.** $S$ est une matrice, donc par linéarité
$$Sy = S(\hat y + r) = S\hat y + Sr = \hat y + Sr. \qquad\blacksquare$$

En d'autres termes, la spline ne touche pas à la partie linéaire de $y$ et ne lisse que
le résidu $r$.

### Solution : le cas de la régression linéaire locale

Même raisonnement : si $y = \hat y$ avec $\hat y_i = a+bx_i$, alors en chaque point
$x_0$ le problème des moindres carrés pondérés locaux admet la solution exacte
$(\alpha,\beta)=(a,b)$ (somme des carrés nulle). La matrice $\mathbf B^TW(x_0)\mathbf B$
étant inversible (au moins deux points de poids non nul), cette solution est unique. Donc
$\hat f(x_0)=a+bx_0$ pour tout $x_0$ : $S\hat y=\hat y$, puis $Sy=\hat y+Sr$ par
linéarité.

Algébriquement : $(\mathbf B^TW\mathbf B)^{-1}\mathbf B^TW\,\mathbf B\,(a,b)^T = (a,b)^T$,
d'où $\hat f(x_0) = (1,x_0)(a,b)^T$.